In [1]:
import os
import pickle
import pickle as pkl
import random
from math import log


In [2]:

import numpy as np
import scipy.sparse as sp



In [3]:
from utils import loadWord2Vec

In [4]:

# build corpus
dataset = 'new2'#DataSet_Misinfo,proppysm
#获取当前脚本所在目录的上一级目录的绝对路径
os.path.abspath(os.path.dirname(os.path.dirname('__file__')))
#os.path.abspath(os.path.dirname(os.path.dirname(__file__)))
os.path.abspath(os.path.dirname(os.getcwd()))
os.path.abspath(os.path.join(os.getcwd(), ".."))
#路径构建
input1 = os.sep.join(['..','..', 'data_tgcn', dataset, dataset+"_new"])
input2 = os.sep.join(['..','..', 'data_tgcn', dataset, 'stanford'])
# input3 = os.sep.join(['..', 'data_tgcn', dataset, 'tbert'])
# input4 = os.sep.join(['..', 'data_tgcn', dataset, 'tbert', dataset])
#input3 = os.sep.join(['..','..', 'data_tgcn', dataset, 'lstm'])
#input4 = os.sep.join(['..','..', 'data_tgcn', dataset, 'lstm', dataset])
input3 = os.sep.join(['..','..', 'data_tgcn', dataset, 'bert'])
input4 = os.sep.join(['..','..', 'data_tgcn', dataset, 'bert', dataset])
output1 = os.sep.join(['..','..', 'data_tgcn', dataset, 'build_train', dataset+"_new"])
output2 = os.sep.join(['..','..', 'data_tgcn', dataset, 'build_train'])
if not os.path.exists(output2):
    os.mkdir(output2)
'''
../../data_tgcn/ptc/ptc_new
../../data_tgcn/ptc/stanford
../../data_tgcn/ptc/bert
../../data_tgcn/ptc/bert/ptc
../../data_tgcn/ptc/build_train/ptc_new
../../data_tgcn/ptc/build_train
'''

'\n../../data_tgcn/ptc/ptc_new\n../../data_tgcn/ptc/stanford\n../../data_tgcn/ptc/bert\n../../data_tgcn/ptc/bert/ptc\n../../data_tgcn/ptc/build_train/ptc_new\n../../data_tgcn/ptc/build_train\n'

In [5]:
#word_embeddings_dim（词嵌入维度）和 window_size（窗口大小）
word_embeddings_dim = 300
window_size = 7

word_vector_map = {}
#https://nlp.stanford.edu/projects/glove/
#GloVe：用于单词表示的全局向量转化为词向量
word_vector_file = '../../glove.6B.300d.txt'
_, embd, word_vector_map = loadWord2Vec(word_vector_file)
word_embeddings_dim = len(embd[0])



Loaded Word Vectors!


In [6]:
# shulffing
doc_name_list = []
doc_train_list = []
doc_test_list = []

f = open(input1 + '_labels.txt', 'r', encoding='ISO-8859-1')
lines = f.readlines()
for line in lines:
    doc_name_list.append(line.strip())
    temp = line.split("\t")
    if temp[1].find('test') != -1:
        doc_test_list.append(line.strip())#所有test的数据条
    elif temp[1].find('train') != -1:
        doc_train_list.append(line.strip())#所有train的数据条
f.close()
#print(lines)
#print(doc_test_list)
#print(doc_train_list)

In [7]:
doc_content_list = []
f = open(input1 + '.clean.txt', 'r')
lines = f.readlines()
for line in lines:
    doc_content_list.append(line.strip())#所有的文本信息
f.close()
#print(doc_content_list)

In [8]:
#print(doc_name_list)
#获取所有的train id，并将其打乱
train_ids = []
for train_name in doc_train_list:
    train_id = doc_name_list.index(train_name)
    train_ids.append(train_id)
random.shuffle(train_ids)

In [9]:

# partial labeled data
# train_ids = train_ids[:int(0.2 * len(train_ids))]#20％

#将列表 train_ids 中的整数索引转换为字符串
train_ids_str = '\n'.join(str(index) for index in train_ids)

#创建ptc_new.train.index并写入训练集的id
f = open(output1 + '.train.index', 'w')#build_train/propaganda2.train.index
f.write(train_ids_str)
f.close()

#创建ptc_new.test.index并写入训练集的id
test_ids = []
for test_name in doc_test_list:
    test_id = doc_name_list.index(test_name)
    test_ids.append(test_id)
# print(test_ids)
random.shuffle(test_ids)

test_ids_str = '\n'.join(str(index) for index in test_ids)
f = open(output1 + '.test.index', 'w')
f.write(test_ids_str)
f.close()

ids = train_ids + test_ids



In [10]:
#打乱后的信息构建
#ptc_new_doc_shuffle.txt打乱后的lable
#ptc_new_word_shuffle.txt打乱后的文本内容
shuffle_doc_name_list = []#随机打乱后的文本信息：id+mode+label
shuffle_doc_words_list = []#随机打乱后的文本信息：句子

for id in ids:
    shuffle_doc_name_list.append(doc_name_list[int(id)])
    shuffle_doc_words_list.append(doc_content_list[int(id)])
shuffle_doc_name_str = '\n'.join(shuffle_doc_name_list)
shuffle_doc_words_str = '\n'.join(shuffle_doc_words_list)

f = open(output1 + '_doc_shuffle.txt', 'w')
f.write(shuffle_doc_name_str)
f.close()

f = open(output1 + '_word_shuffle.txt', 'w')
f.write(shuffle_doc_words_str)
f.close()


In [11]:
#构建词汇表vocab（包含所有不重复单词的列表）
#计算每个单词在文档中出现的频率（word_freq）
word_freq = {}
word_set = set()
for doc_words in shuffle_doc_words_list:
    words = doc_words.split()
    for word in words:
        word_set.add(word)
        if word in word_freq:
            word_freq[word] += 1
        else:
            word_freq[word] = 1

vocab = list(word_set)
vocab_size = len(vocab)

#print(word_freq)
#print(word_set)
#print(len(vocab))

In [12]:
# word_doc_list每个单词在哪个句子中出现
word_doc_list = {}

for i in range(len(shuffle_doc_words_list)):
    doc_words = shuffle_doc_words_list[i]
    words = doc_words.split()
    appeared = set()
    for word in words:
        if word in appeared:
            continue#继续循环但不执行以下操作
        if word in word_doc_list:
            doc_list = word_doc_list[word]
            doc_list.append(i)
            word_doc_list[word] = doc_list
        else:
            word_doc_list[word] = [i]
        appeared.add(word)

#word_doc_freq每个单词在全文中的几个句子中出现过
word_doc_freq = {}
for word, doc_list in word_doc_list.items():#key,value
    word_doc_freq[word] = len(doc_list)


In [13]:
#将词汇表转换为字符串并保存到文件
word_id_map = {}
id_word_map = {}
for i in range(vocab_size):
    word_id_map[vocab[i]] = i
    id_word_map[i] = vocab[i]
vocab_str = '\n'.join(vocab)

f = open(output1 + '_vocab.txt', 'w')
f.write(vocab_str)
f.close()

In [14]:
# label list
#所有的标签的种类（0,1）
label_set = set()
for doc_meta in shuffle_doc_name_list:
    temp = doc_meta.split('\t')
    label_set.add(temp[2])
label_list = list(label_set)

#将信息写入ptc_new_labels
label_list_str = '\n'.join(label_list)
f = open(output1 + '_labels.txt', 'w')
f.write(label_list_str)
f.close()

In [15]:
'''
构建features
'''

train_size = len(train_ids)
val_size = int(0.1 * train_size)
real_train_size = train_size - val_size  # - int(0.5 * train_size)
test_size=len(test_ids)

print(real_train_size,val_size,test_size)


11305 1256 6702


In [16]:
#真实参与训练的数据train*(1-0.1),0.1的train转换为val
real_train_doc_names = shuffle_doc_name_list[:real_train_size]
real_train_doc_names_str = '\n'.join(real_train_doc_names)

f = open(output1 + '.real_train.name', 'w')
f.write(real_train_doc_names_str)
f.close()

In [17]:
#创建训练集的一个稀疏矩阵 x，一个标签向量 y
row_x = []
col_x = []
data_x = []
for i in range(real_train_size):
    doc_vec = np.array([0.0 for k in range(word_embeddings_dim)])#初始化word_embeddings_dim维的向量
    doc_words = shuffle_doc_words_list[i]#句子
    words = doc_words.split()
    doc_len = len(words)
    for word in words:
        if word in word_vector_map:
            word_vector = word_vector_map[word]
            doc_vec = doc_vec + np.array(word_vector)

    for j in range(word_embeddings_dim):
        row_x.append(i)
        col_x.append(j)
        # np.random.uniform(-0.25, 0.25)
        data_x.append(doc_vec[j] / doc_len)  # doc_vec[j]/ doc_len

# x = sp.csr_matrix((real_train_size, word_embeddings_dim), dtype=np.float32)
x = sp.csr_matrix((data_x, (row_x, col_x)), shape=(
    real_train_size, word_embeddings_dim))

y = []
for i in range(real_train_size):
    doc_meta = shuffle_doc_name_list[i]
    temp = doc_meta.split('\t')
    label = temp[2]
    one_hot = [0 for l in range(len(label_list))]
    label_index = label_list.index(label)
    one_hot[label_index] = 1
    y.append(one_hot)
y = np.array(y)
# print(y)


In [18]:
# tx: feature vectors of test docs, no initial features创建测试集的一个稀疏矩阵 tx，一个标签向量 ty没有初始特征
test_size = len(test_ids)

row_tx = []
col_tx = []
data_tx = []
for i in range(test_size):
    doc_vec = np.array([0.0 for k in range(word_embeddings_dim)])
    doc_words = shuffle_doc_words_list[i + train_size]
    words = doc_words.split()
    doc_len = len(words)
    for word in words:
        if word in word_vector_map:
            word_vector = word_vector_map[word]
            doc_vec = doc_vec + np.array(word_vector)

    for j in range(word_embeddings_dim):
        row_tx.append(i)
        col_tx.append(j)
        # np.random.uniform(-0.25, 0.25)
        data_tx.append(doc_vec[j] / doc_len)  # doc_vec[j] / doc_len

# tx = sp.csr_matrix((test_size, word_embeddings_dim), dtype=np.float32)
tx = sp.csr_matrix((data_tx, (row_tx, col_tx)),
                   shape=(test_size, word_embeddings_dim))

ty = []
for i in range(test_size):
    doc_meta = shuffle_doc_name_list[i + train_size]
    temp = doc_meta.split('\t')
    label = temp[2]
    one_hot = [0 for l in range(len(label_list))]
    label_index = label_list.index(label)
    one_hot[label_index] = 1
    ty.append(one_hot)
ty = np.array(ty)
# print(ty)


In [19]:
# allx: 标记和未标记的训练实例的特征向量
# (a superset of x)
# unlabeled train instances -> words 未标记的训练

word_vectors = np.random.uniform(-0.01, 0.01,
                                 (vocab_size, word_embeddings_dim))

#根据给定的 word_vector_map 更新或初始化一个 word_vectors 矩阵，使得其中的每个单词向量都能根据 word_vector_map 中的值进行更新。
#利用外部提供的词向量信息glove.6B.300d.txt'来改进模型的输入表示，从而提高模型在各种自然语言处理任务中的性能。
#vocab所有单词
for i in range(len(vocab)):
    word = vocab[i]
    if word in word_vector_map:
        vector = word_vector_map[word]
        word_vectors[i] = vector


In [20]:
row_allx = []
col_allx = []
data_allx = []

#训练集中的每个文档创建一个特征向量，并将这些特征向量存储在稀疏矩阵的形式中
for i in range(train_size):
    #初始化空特征向量和处理文档：
    doc_vec = np.array([0.0 for k in range(word_embeddings_dim)])
    doc_words = shuffle_doc_words_list[i]
    words = doc_words.split()
    doc_len = len(words)
    #累加单词向量到特征向量：
    for word in words:
        if word in word_vector_map:
            word_vector = word_vector_map[word]
            doc_vec = doc_vec + np.array(word_vector)
    
    #构建稀疏矩阵的行索引、列索引和数据：
    for j in range(word_embeddings_dim):
        row_allx.append(int(i))
        col_allx.append(j)
        # np.random.uniform(-0.25, 0.25)
        data_allx.append(doc_vec[j] / doc_len)  # doc_vec[j]/doc_len
#将词汇表中每个单词的预训练词向量添加到稀疏矩阵的数据中
for i in range(vocab_size):
    for j in range(word_embeddings_dim):
        row_allx.append(int(i + train_size))
        col_allx.append(j)
        data_allx.append(word_vectors.item((i, j)))
#转换为 NumPy 数组

row_allx = np.array(row_allx)
col_allx = np.array(col_allx)
data_allx = np.array(data_allx)

In [21]:
print(row_allx)
print(col_allx)
print(data_allx)
'''
row_allx存储了用于表示每个节点（或实例）的特征向量的索引。
col_allx指示稀疏矩阵 allx 中每个非零元素所在列的索引列表。
data_allx存储稀疏矩阵 allx 中每个非零元素的特征值
'''

[    0     0     0 ... 22257 22257 22257]
[  0   1   2 ... 297 298 299]
[-0.04356267  0.116502   -0.01704592 ... -0.9178     -0.57154
  0.42676   ]


'\nrow_allx存储了用于表示每个节点（或实例）的特征向量的索引。\ncol_allx指示稀疏矩阵 allx 中每个非零元素所在列的索引列表。\ndata_allx存储稀疏矩阵 allx 中每个非零元素的特征值\n'

In [22]:
#data_allx: 是一个一维数组，包含了要放入稀疏矩阵中的数据值。(row_allx, col_allx): 这是一个元组，包含了两个数组 row_allx 和 col_allx，分别表示数据值的行索引和列索引。
#hape=(train_size + vocab_size, word_embeddings_dim) 指定了稀疏矩阵的形状，行数为 train_size + vocab_size，列数为 word_embeddings_dim
allx = sp.csr_matrix(
    (data_allx, (row_allx, col_allx)), shape=(train_size + vocab_size, word_embeddings_dim))

ally = []
for i in range(train_size):
    doc_meta = shuffle_doc_name_list[i]
    temp = doc_meta.split('\t')
    label = temp[2]
    one_hot = [0 for l in range(len(label_list))]
    label_index = label_list.index(label)
    one_hot[label_index] = 1
    ally.append(one_hot)

for i in range(vocab_size):
    one_hot = [0 for l in range(len(label_list))]
    ally.append(one_hot)

ally = np.array(ally)

print(x.shape, y.shape, tx.shape, ty.shape, allx.shape, ally.shape)

(11305, 300) (11305, 2) (6702, 300) (6702, 2) (22258, 300) (22258, 2)


In [23]:
# dump objects
f = open(output2 + "/ind.{}.x".format(dataset+"_new"), 'wb')
pkl.dump(x, f)
f.close()

f = open(output2 + "/ind.{}.y".format(dataset+"_new"), 'wb')
pkl.dump(y, f)
f.close()

f = open(output2 + "/ind.{}.tx".format(dataset+"_new"), 'wb')
pkl.dump(tx, f)
f.close()

f = open(output2 + "/ind.{}.ty".format(dataset+"_new"), 'wb')
pkl.dump(ty, f)
f.close()

f = open(output2 + "/ind.{}.allx".format(dataset+"_new"), 'wb')
pkl.dump(allx, f)
f.close()

f = open(output2 + "/ind.{}.ally".format(dataset+"_new"), 'wb')
pkl.dump(ally, f)
f.close()

In [24]:
'''
Doc word heterogeneous graph_only_2.txt 1
'''
#顺序特征
# word co-occurence with context windows
windows = []#对数据进行处理，仅取单词数量小于等于7的单词，超过则使用滑动窗口一次向前移动一个单词的距离
'''
['liberals', 'agree', 'trump', 'tougher', 'putin', 'obama', 'leftwing', 'journalist', 'glenn', 'greenwald', 'fan', 'president', 'trump', 'policies', ',', 'simply', 'cannot', 'understand', 'leftwing', 'media', 'keeps', 'getting', 'everything', 'trump']
*****************************
['liberals', 'agree', 'trump', 'tougher', 'putin', 'obama', 'leftwing']
['agree', 'trump', 'tougher', 'putin', 'obama', 'leftwing', 'journalist']
['trump', 'tougher', 'putin', 'obama', 'leftwing', 'journalist', 'glenn']
['tougher', 'putin', 'obama', 'leftwing', 'journalist', 'glenn', 'greenwald']
['putin', 'obama', 'leftwing', 'journalist', 'glenn', 'greenwald', 'fan']
['obama', 'leftwing', 'journalist', 'glenn', 'greenwald', 'fan', 'president']
['leftwing', 'journalist', 'glenn', 'greenwald', 'fan', 'president', 'trump']
['journalist', 'glenn', 'greenwald', 'fan', 'president', 'trump', 'policies']
['glenn', 'greenwald', 'fan', 'president', 'trump', 'policies', ',']
['greenwald', 'fan', 'president', 'trump', 'policies', ',', 'simply']
['fan', 'president', 'trump', 'policies', ',', 'simply', 'cannot']
['president', 'trump', 'policies', ',', 'simply', 'cannot', 'understand']
['trump', 'policies', ',', 'simply', 'cannot', 'understand', 'leftwing']
['policies', ',', 'simply', 'cannot', 'understand', 'leftwing', 'media']
[',', 'simply', 'cannot', 'understand', 'leftwing', 'media', 'keeps']
['simply', 'cannot', 'understand', 'leftwing', 'media', 'keeps', 'getting']
['cannot', 'understand', 'leftwing', 'media', 'keeps', 'getting', 'everything']
['understand', 'leftwing', 'media', 'keeps', 'getting', 'everything', 'trump']
'''
for doc_words in shuffle_doc_words_list:
    words = doc_words.split()
    length = len(words)
    if length <= window_size:
        windows.append(words)
    else:
        #print(length, length - window_size + 1)
        for j in range(length - window_size + 1):
            window = words[j: j + window_size]
            windows.append(window)
            #print(window)
word_window_freq = {}#每个单词在windows中出现的次数
for window in windows:
    appeared = set()
    for i in range(len(window)):
        if window[i] in appeared:
            continue
        if window[i] in word_window_freq:
            word_window_freq[window[i]] += 1
        else:
            word_window_freq[window[i]] = 1
        appeared.add(window[i])

word_pair_count = {}#每对单词在整个语料库中的共现次数
for window in windows:
    for i in range(1, len(window)):
        for j in range(0, i):
            word_i = window[i]#单词
            word_i_id = word_id_map[word_i]#该单词的序号
            word_j = window[j]#前面的单词
            word_j_id = word_id_map[word_j]#前面单词的序号
            if word_i_id == word_j_id:#如果单词相同则不进行处理
                continue
            word_pair_str = str(word_i_id) + ',' + str(word_j_id)
            if word_pair_str in word_pair_count:#两个单词组成单词对，word_pair_count记录该单词对在语料库中出现的次数
                word_pair_count[word_pair_str] += 1
            else:
                word_pair_count[word_pair_str] = 1
            # two orders
            word_pair_str = str(word_j_id) + ',' + str(word_i_id)
            if word_pair_str in word_pair_count:
                word_pair_count[word_pair_str] += 1
            else:
                word_pair_count[word_pair_str] = 1

In [25]:
row = []
col = []
weight = []
weight1 = []
weight2 = []

# 根据stanford句法依存构建边权重
data1 = pickle.load(open(input2 + "/{}_stan.pkl".format(dataset), "rb"))
max_count1 = 0.0
min_count1 = 0.0
count1 = []
for key in data1:
    if data1[key] > max_count1:
        max_count1 = data1[key]
    if data1[key] < min_count1:
        min_count1 = data1[key]
    count1.append(data1[key])#添加value值
count_mean1 = np.mean(count1)
count_var1 = np.var(count1)
count_std1 = np.std(count1, ddof=1)

# 根据语义依存构建边权重
data2 = pickle.load(open(input4 + "_semantic.pkl", "rb"))#..\\data_tgcn\\mr\\bert\\mr_semantic_0.05.pkl'mr_semantic.pkl
max_count2 = 0.0
min_count2 = 10.0
#min_count2 = 0.0
count2 = []
for key in data2:
    #key:('next',),('next',)
    #data[key]:次数
    if data2[key] > max_count2:
        max_count2 = data2[key]
    if data2[key] < min_count2:
        min_count2 = data2[key]
    count2.append(data2[key])
count_mean2 = np.mean(count2)#平均
count_var2 = np.var(count2)#方差
count_std2 = np.std(count2, ddof=1)#标准差

# compute weights
num_window = len(windows)
for key in word_pair_count:
    temp = key.split(',')
    i = int(temp[0])
    j = int(temp[1])
    count = word_pair_count[key]#单词对在语料库中出现的次数
    word_freq_i = word_window_freq[vocab[i]]#第一个单词在语料库中出现的次数
    word_freq_j = word_window_freq[vocab[j]]#第二个单词在语料库中出现的次数

    #计算互信息（PMI）
    pmi = log((1.0 * count / num_window) /
              (1.0 * word_freq_i * word_freq_j / (num_window * num_window)))
    if pmi <= 0:
        continue
    # pmi
    row.append(train_size + i)
    col.append(train_size + j)
    weight.append(pmi)

    # 句法依存
    if i not in id_word_map or j not in id_word_map:
        continue
    newkey = id_word_map[i] + ',' + id_word_map[j]#将序号转换为word
    if newkey in data1:
        # min-max标准化
        wei = (data1[newkey] - min_count1) / (max_count1 - min_count1)
        # 0均值标准化
        # wei = (data1[key]-count_mean1)/ count_std1
        # 出现频度比例，出现1的时候比较多
        # wei = data1[key] / data2[key]
        weight1.append(wei)
    else:
        weight1.append(0)

    # 语义依存
    if newkey in data2:
        # min-max标准化
        wei = (data2[newkey] - min_count2) / (max_count2 - min_count2)
        # 0均值标准化
        # wei = (data2[key]-count_mean2)/ count_std2
        # 出现频度比例，出现1的时候比较多
        # wei = data2[key] / data2[key]
        weight2.append(wei)
    else:
        weight2.append(0)

'''
for i in range(vocab_size):
    for j in range(vocab_size):
        if vocab[i] in word_vector_map and vocab[j] in word_vector_map:
            vector_i = np.array(word_vector_map[vocab[i]])
            vector_j = np.array(word_vector_map[vocab[j]])
            similarity = 1.0 - cosine(vector_i, vector_j)
            if similarity > 0.9:
                print(vocab[i], vocab[j], similarity)
                row.append(train_size + i)
                col.append(train_size + j)
                weight.append(similarity)
'''
# doc word frequency
weight_tfidf = []
doc_word_freq = {}
for doc_id in range(len(shuffle_doc_words_list)):
    doc_words = shuffle_doc_words_list[doc_id]
    words = doc_words.split()
    for word in words:
        word_id = word_id_map[word]
        doc_word_str = str(doc_id) + ',' + str(word_id)
        if doc_word_str in doc_word_freq:
            doc_word_freq[doc_word_str] += 1
        else:
            doc_word_freq[doc_word_str] = 1

for i in range(len(shuffle_doc_words_list)):
    doc_words = shuffle_doc_words_list[i]
    words = doc_words.split()
    doc_word_set = set()
    for word in words:
        if word in doc_word_set:
            continue
        j = word_id_map[word]
        key = str(i) + ',' + str(j)
        freq = doc_word_freq[key]
        if i < train_size:
            row.append(i)
        else:
            row.append(i + vocab_size)
        col.append(train_size + j)
        idf = log(1.0 * len(shuffle_doc_words_list) /
                  word_doc_freq[vocab[j]])
        weight_tfidf.append(freq * idf)
        doc_word_set.add(word)

weight = weight + weight_tfidf
node_size = train_size + vocab_size + test_size
adj = sp.csr_matrix(
    (weight, (row, col)), shape=(node_size, node_size))


# dump objects
f = open(output2 + '/ind.{}.adj'.format(dataset+"_new"), 'wb')
pkl.dump(adj, f)
f.close()
import collections
d=collections.defaultdict(list)
for i in range(0,len(row)):
    d[row[i]].append(col[i])

f = open(output2 + '/ind.{}.graph'.format(dataset+"_new"), 'wb')
pkl.dump(d, f)
f.close()

print('构图1完成')

weight1 = weight1 + weight_tfidf
node_size = train_size + vocab_size + test_size
adj = sp.csr_matrix(
    (weight1, (row, col)), shape=(node_size, node_size))

# dump objects
f = open(output2 + '/ind.{}.adj1'.format(dataset+"_new"), 'wb')
pkl.dump(adj, f)
f.close()

d1=collections.defaultdict(list)
for i in range(0,len(row)):
    d1[row[i]].append(col[i])

f = open(output2 + '/ind.{}.graph1'.format(dataset+"_new"), 'wb')
pkl.dump(d1, f)
f.close()
print(adj)
print('构图2完成')

weight2 = weight2 + weight_tfidf
node_size = train_size + vocab_size + test_size
adj = sp.csr_matrix(
    (weight2, (row, col)), shape=(node_size, node_size))

# dump objects
f = open(output2 + '/ind.{}.adj2'.format(dataset+"_new"), 'wb')
pkl.dump(adj, f)
f.close()

d2 = collections.defaultdict(list)
for i in range(0, len(row)):
    d2[row[i]].append(col[i])

f = open(output2 + '/ind.{}.graph2'.format(dataset+"_new"), 'wb')
pkl.dump(d2, f)
f.close()
print('构图3完成')


构图1完成
  (0, 16086)	4.393670762790683
  (0, 17069)	0.8050208736076271
  (0, 17077)	8.256503524028057
  (0, 17985)	6.2550235238179335
  (0, 17990)	4.656455283620737
  (0, 18034)	4.778345101229774
  (0, 19965)	5.561876343257988
  (0, 20298)	5.129742988067663
  (0, 21184)	6.607844898440676
  (0, 21864)	5.934115803737832
  (0, 22007)	6.15236936975785
  (1, 13456)	3.7633828418485895
  (1, 13994)	6.081751802543897
  (1, 14051)	4.142356334509778
  (1, 14580)	2.8784511894611673
  (1, 15506)	5.548453322925847
  (1, 15967)	5.876957389897884
  (1, 16071)	4.718446959648705
  (1, 18816)	5.129742988067663
  (1, 19143)	4.340488497330374
  (1, 19293)	4.034058959178641
  (1, 21464)	5.589275317446103
  (1, 21582)	2.7004479614013124
  (2, 12581)	5.617446194412799
  (2, 13366)	4.336512348950735
  :	:
  (28958, 17308)	4.538065267672577
  (28958, 17576)	4.953286550726106
  (28958, 17949)	4.651005678853172
  (28958, 18033)	5.301593244994322
  (28958, 18071)	6.128271818178789
  (28958, 18842)	4.320763991982596

In [26]:
def print_graph_shape(graph, graph_name):
    node_count = len(graph)  # 图中的节点数
    edge_count = sum(len(edges) for edges in graph.values())  # 所有节点的邻接节点数之和，即图的边数
    print(f"{graph_name} 的形状：节点数 = {node_count}, 边数 = {edge_count}")

# 假设您有三个图 d, d1, d2 存储在 defaultdict 中
print_graph_shape(d, "第一个图")
print_graph_shape(d1, "第二个图")
print_graph_shape(d2, "第三个图")


第一个图 的形状：节点数 = 28960, 边数 = 1376019
第二个图 的形状：节点数 = 28960, 边数 = 1376019
第三个图 的形状：节点数 = 28960, 边数 = 1376019
